#### Task 3: Simple Modular,  Logged ETL System

This notebook keeps the ETL flow simple: extract data from a public API, clean and transform it, print a small summary table, and load the result into CSV and MySQL.

In [1]:
import pandas as pd
import requests
import mysql.connector
from datetime import datetime


def log(message):
    current_time = datetime.now().strftime("%H:%M:%S")
    print(f"[{current_time}] {message}")

In [2]:
def extract():
    log("Starting extract()")

    url = "https://jsonplaceholder.typicode.com/posts"

    try:
        response = requests.get(url, timeout=5)
        response.raise_for_status()
        data = response.json()
        log(f"Extracted {len(data)} rows from API")
        return data

    except requests.exceptions.Timeout:
        log("Timeout Error while fetching API")
    except requests.exceptions.HTTPError as e:
        log(f"HTTP Error: {e}")
    except requests.exceptions.RequestException as e:
        log(f"Request Error: {e}")

    return None

In [3]:
def clean(data):
    log("Starting clean()")

    df = pd.DataFrame(data)
    input_rows = len(df)

    df = df[['userId', 'id', 'title', 'body']]
    df = df.drop_duplicates(subset=['id'])
    df = df.dropna()

    df['title'] = df['title'].str.strip().str.lower()
    df['body'] = df['body'].str.strip()

    output_rows = len(df)
    log(f"Cleaned rows: {input_rows} -> {output_rows}")

    return df

In [4]:
def transform(df):
    log("Starting transform()")

    input_rows = len(df)

    df['word_count'] = df['title'].str.split().str.len()
    df['body_length'] = df['body'].str.len()
    df['title_category'] = df['word_count'].apply(lambda x: 'Short' if x <= 3 else 'Long')

    log("Added 3 new calculated columns")

    summary = df.groupby('title_category')['word_count'].agg(['mean', 'min', 'max'])

    print("\nSummary Table:")
    print(summary)

    output_rows = len(df)
    log(f"Transform rows: {input_rows} -> {output_rows}")

    return df

In [ ]:
def load(df):
    log("Starting load()")

    df.to_csv("final_posts.csv", index=False)
    log("CSV file saved")

    conn = None
    cursor = None

    try:
        conn = mysql.connector.connect(
            host="localhost",
            user="root",
            password="root"
        )

        cursor = conn.cursor()

        cursor.execute("""
            CREATE DATABASE IF NOT EXISTS etl_db
        """)

        cursor.execute("USE etl_db")

        cursor.execute("""
            CREATE TABLE IF NOT EXISTS posts (
                id INT PRIMARY KEY,
                userId INT,
                title TEXT,
                body TEXT,
                word_count INT,
                body_length INT,
                title_category VARCHAR(20)
            )
        """)

        query = """
        INSERT INTO posts
        (id, userId, title, body, word_count, body_length, title_category)

        VALUES (%s, %s, %s, %s, %s, %s, %s)

        ON DUPLICATE KEY UPDATE
            title = VALUES(title),
            body = VALUES(body),
            word_count = VALUES(word_count),
            body_length = VALUES(body_length),
            title_category = VALUES(title_category)
        """

        for _, row in df.iterrows():
            cursor.execute(query, (
                int(row['id']),
                int(row['userId']),
                row['title'],
                row['body'],
                int(row['word_count']),
                int(row['body_length']),
                row['title_category']
            ))

        conn.commit()
        log(f"Loaded {len(df)} rows into MySQL")

    except mysql.connector.Error as e:
        log(f"MySQL Error: {e}")

    finally:
        if cursor:
            cursor.close()
        if conn and conn.is_connected():
            conn.close()
        log("MySQL connection closed")